# vector-normalize-keepdim — worked example 3: Column-normalize a weight matrix with eps guard

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `vector-normalize-keepdim`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

When normalizing matrix columns for spectral normalization or dictionary learning, we compute per-column norms with `dim=0, keepdim=True` and add a small `eps` to prevent division by zero. The `keepdim=True` gives shape `(1, M)` for an `(N, M)` input, enabling broadcast division over the N rows.

## Worked solution

**Step 1 — identify the axis.**
For a weight matrix of shape `(N, M)`, each column corresponds to a `dim=0` slice. We want each column to have unit L2 norm, so we reduce along `dim=0`.

**Step 2 — compute norms safely.**
We call `W.norm(dim=0, keepdim=True)`, producing shape `(1, M)` — one scalar per column. We add `eps=1e-8` to guard against zero columns (which would produce NaN in division).

**Step 3 — broadcast divide.**
Dividing W (shape `(N, M)`) by the guarded norms (shape `(1, M)`) broadcasts across the N rows. Each column of the result has L2 norm ≈ 1.0 (exactly 1.0 for non-zero columns when eps is negligible).

**Step 4 — verify.**
Compute `.norm(dim=0)` on the result and confirm values close to 1.

In [ ]:
import torch as t

t.manual_seed(99)
N, M = 12, 8   # 12-dimensional vectors, 8 columns
eps = 1e-8

W = t.randn(N, M)

# Step 1: per-column norms, keepdim so shape stays (1, M) not (M,)
col_norms = W.norm(dim=0, keepdim=True)        # (1, M)

# Step 2: guarded divide
W_normalized = W / (col_norms + eps)           # (N, M)

# Verify
result_col_norms = W_normalized.norm(dim=0)    # (M,)
print('Column norms (all should be ~1.0):', result_col_norms.tolist())
print('Max deviation:', (result_col_norms - 1.0).abs().max().item())